# Notebook 06 — Hybrid HQNN (Quantum + Classical ReLU)

**Mechanism:** Offload non-linearity to a classical post-processing layer.
The quantum circuit acts as a linear feature extractor; a dense layer with ReLU
introduces the non-linearity classically. This is the simplest hybrid baseline.

**Architecture:**
```
x  →  [VQC: n_qubits × n_layers]  →  expectation values  →  [Linear(m → 1) + ReLU]  →  ŷ
```

Multiple qubit expectation values `<Z_i>` are collected as a feature vector,
then passed through a small classical MLP.

**Role in QNLEP:** Upper bound on what a hybrid model achieves at the same
qubit cost. If HQNN >> purely quantum methods, the non-linearity gap is real.
If scores are similar, quantum non-linearity mechanisms are competitive.

**Also includes:** a classical MLP baseline (no quantum component) with matched
parameter count — the floor reference for the entire QNLEP suite.

In [ ]:
import sys
sys.path.insert(0, "..")

import os
import numpy as np
import pennylane as qml
from pennylane import numpy as pnp
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
import json, pathlib

from src.qnlep_run_paths import protocol_results_dir, protocol_n_qubits, protocol_n_layers

RESULTS_DIR = protocol_results_dir()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RUN_SEED = int(os.environ.get("QNLEP_SEED", "42"))
np.random.seed(RUN_SEED)
torch.manual_seed(RUN_SEED)

## 1. Configuration

In [ ]:
N_QUBITS      = protocol_n_qubits(3)
N_LAYERS      = protocol_n_layers(3)
CLASSICAL_DIM = 16   # hidden units in classical MLP head
N_TRAIN       = 30
N_EVAL        = 100
N_STEPS       = 80
LR            = 1e-2

x_train = np.linspace(-np.pi, np.pi, N_TRAIN)
x_eval  = np.linspace(-np.pi, np.pi, N_EVAL)
y_train = np.sin(x_train)
y_eval  = np.sin(x_eval)

X_tr = torch.tensor(x_train, dtype=torch.float32).unsqueeze(1)
Y_tr = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_ev = torch.tensor(x_eval,  dtype=torch.float32).unsqueeze(1)

## 2. Quantum feature extractor

Returns a vector of `n_qubits` expectation values `[<Z_0>, <Z_1>, ..., <Z_{n-1}>]`
instead of a single scalar. This is the quantum 'embedding layer'.

In [ ]:
dev = qml.device('default.qubit', wires=N_QUBITS)

@qml.qnode(dev, interface='torch', diff_method='backprop')
def quantum_embedding(x, q_weights):
    """
    Data re-uploading circuit that returns <Z_i> for all qubits.
    q_weights shape: (N_LAYERS, N_QUBITS, 2).
    """
    for l in range(N_LAYERS):
        for w in range(N_QUBITS):
            qml.RY(x, wires=w)                    # encode x
            qml.RY(q_weights[l, w, 0], wires=w)   # trainable rotation
            qml.RZ(q_weights[l, w, 1], wires=w)
        for w in range(N_QUBITS - 1):
            qml.CNOT(wires=[w, w + 1])             # entanglement
    return [qml.expval(qml.PauliZ(w)) for w in range(N_QUBITS)]

## 3. Hybrid HQNN model (PyTorch nn.Module)

In [ ]:
class HybridHQNN(nn.Module):
    def __init__(self, n_qubits=N_QUBITS, n_layers=N_LAYERS, hidden=CLASSICAL_DIM):
        super().__init__()
        self.n_qubits  = n_qubits
        self.n_layers  = n_layers

        # quantum parameters (trainable)
        self.q_weights = nn.Parameter(
            torch.tensor(np.random.normal(0, 0.1, (n_layers, n_qubits, 2)),
                         dtype=torch.float32)
        )

        # classical MLP head: n_qubits features → hidden → 1
        self.classical_head = nn.Sequential(
            nn.Linear(n_qubits, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1)
        )

    def forward(self, x_batch):
        # x_batch: (batch, 1)
        features = torch.stack([
            torch.stack(quantum_embedding(x_batch[i, 0], self.q_weights)).float()
            for i in range(x_batch.shape[0])
        ])  # (batch, n_qubits)
        return self.classical_head(features)  # (batch, 1)

## 4. Classical MLP baseline (matched parameter count)

In [ ]:
def count_params(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

hqnn    = HybridHQNN()
n_hqnn  = count_params(hqnn)

# Build a classical MLP with approximately the same parameter count
# 1 → h → h → 1 where h is chosen so total params ≈ n_hqnn
import math
h_classical = max(4, int((-3 + math.sqrt(9 + 4 * (n_hqnn - 1))) / 2))

classical_mlp = nn.Sequential(
    nn.Linear(1, h_classical),
    nn.ReLU(),
    nn.Linear(h_classical, h_classical),
    nn.ReLU(),
    nn.Linear(h_classical, 1)
)

print(f"HQNN params:     {n_hqnn}")
print(f"Classical MLP:   {count_params(classical_mlp)} params  (hidden={h_classical})")

## 5. Training loop (shared)

In [ ]:
def train_torch_model(model, X, Y, n_steps=N_STEPS, lr=LR):
    opt      = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn  = nn.MSELoss()
    history  = []
    for step in range(n_steps):
        opt.zero_grad()
        pred = model(X)
        loss = loss_fn(pred, Y)
        loss.backward()
        opt.step()
        history.append(loss.item())
        if step % 20 == 0:
            print(f"  step {step:3d}  loss={loss.item():.4f}")
    return history

print('--- Hybrid HQNN ---')
hist_hqnn = train_torch_model(hqnn, X_tr, Y_tr)

print('--- Classical MLP ---')
hist_mlp = train_torch_model(classical_mlp, X_tr, Y_tr)

## 6. Evaluate

In [ ]:
with torch.no_grad():
    preds_hqnn = hqnn(X_ev).numpy().ravel()
    preds_mlp  = classical_mlp(X_ev).numpy().ravel()

r2_hqnn = r2_score(y_eval, preds_hqnn)
r2_mlp  = r2_score(y_eval, preds_mlp)

print(f"Hybrid HQNN    R²={r2_hqnn:.4f}  MSE={np.mean((preds_hqnn - y_eval)**2):.4f}")
print(f"Classical MLP  R²={r2_mlp:.4f}  MSE={np.mean((preds_mlp  - y_eval)**2):.4f}")

## 7. Visualizations

### 7a. Training curves — both models

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3))
ax.semilogy(hist_hqnn, label=f'Hybrid HQNN ({n_hqnn} params)', color='C2')
ax.semilogy(hist_mlp,  label=f'Classical MLP ({count_params(classical_mlp)} params)', color='C3', ls='--')
ax.set_xlabel('Step')
ax.set_ylabel('MSE (log scale)')
ax.set_title('Training curves — HQNN vs classical MLP')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/hqnn_training.pdf', bbox_inches='tight')
plt.show()

### 7b. Fit quality comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5), sharey=True)
for ax, preds, label, color in zip(
    axes,
    [preds_hqnn, preds_mlp],
    [f'Hybrid HQNN  R²={r2_hqnn:.3f}', f'Classical MLP  R²={r2_mlp:.3f}'],
    ['C2', 'C3']
):
    ax.plot(x_eval, y_eval,    'k--', lw=1.5, label='target')
    ax.plot(x_eval, preds,      color, lw=2,   label=label)
    ax.scatter(x_train, y_train, color='k', s=15, zorder=3)
    ax.set_xlabel('x')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('f(x)')
fig.suptitle('Hybrid HQNN vs Classical MLP — sin(x)', y=1.02)
plt.tight_layout()
plt.savefig('../figures/hqnn_fit.pdf', bbox_inches='tight')
plt.show()

### 7c. Architecture diagram (text)

In [ ]:
print("Hybrid HQNN architecture:")
print(f"  Input x (scalar)")
print(f"  └─ VQC quantum embedding: {N_QUBITS} qubits × {N_LAYERS} layers")
print(f"       [RY(x) + RY(θ) + RZ(φ) + CNOT chain] × {N_LAYERS}")
print(f"       → [{N_QUBITS} expectation values <Z_i>]")
print(f"  └─ Classical head:")
print(f"       Linear({N_QUBITS} → {CLASSICAL_DIM}) → ReLU → Linear({CLASSICAL_DIM} → 1)")
print(f"  Total trainable params: {n_hqnn}")
print()
print("Classical MLP baseline architecture:")
print(f"  Input x (scalar)")
print(f"  └─ Linear(1 → {h_classical}) → ReLU → Linear({h_classical} → {h_classical}) → ReLU → Linear({h_classical} → 1)")
print(f"  Total trainable params: {count_params(classical_mlp)}")

## 8. Export for downstream notebooks

In [ ]:
import json
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Wrap HQNN as a PennyLane-compatible callable for notebooks 03/04/07
def build_hybrid(x, params, n_qubits=N_QUBITS, n_layers=N_LAYERS):
    """Adapter: params = q_weights numpy array; classical head uses stored weights."""
    with torch.no_grad():
        hqnn.q_weights.data = torch.tensor(params, dtype=torch.float32)
        return float(hqnn(torch.tensor([[x]], dtype=torch.float32)).item())

torch.save(hqnn.state_dict(), RESULTS_DIR / 'hqnn_weights.pt')
torch.save(classical_mlp.state_dict(), RESULTS_DIR / 'mlp_weights.pt')

baseline_scores = {'hybrid_hqnn': {'r2': r2_hqnn}, 'classical_mlp': {'r2': r2_mlp}}
with open(RESULTS_DIR / 'baseline_scores.json', 'w') as f:
    json.dump(baseline_scores, f, indent=2)
print('Saved hqnn_weights.pt, mlp_weights.pt, baseline_scores.json')